# Model and Dataset Paths

In [ ]:
model_repo_path = r"HuggingFaceTB/SmolLM2-135M"
data_set_path = r"yahma/alpaca-cleaned"

# Checking Torch Version and Device Setting

In [ ]:
import torch

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
device, torch.__version__

# Loading Secrets

In [ ]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# secret_value_0 = user_secrets.get_secret("HF_TOKEN")
import os
from dotenv import load_dotenv
load_dotenv()
secret_value_0 = os.getenv('HF_TOKEN')

In [ ]:
import huggingface_hub

huggingface_hub.login(token=secret_value_0)

# Downloadind Datasets from HuggingFace

In [ ]:
from datasets import load_dataset

dataset = load_dataset(path=data_set_path, cache_dir="data/")
dataset

# Sampling few examples

In [ ]:
import random
samples = dataset['train'][random.sample(range(len(dataset['train'])), k=5)]
samples

# Initializing Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_repo_path, cache_dir='tokenizer/')
tokenizer.pad_token = tokenizer.eos_token # using eos token as pad token
print(tokenizer.vocab_size)

In [ ]:
tokenizer(["Hi there", "Hi there, how are u?"], return_tensors="pt", tokenize=False, padding=True, truncation=True)

In [ ]:
def encode_example(example):
    example['encoded_instruction'] = tokenizer(example['instruction'], return_tensors="pt")
    example['encoded_respone'] = tokenizer(example['output'], return_tensors="pt")
    return example

In [ ]:
tokenizer('<|endoftext|>'), tokenizer('<|im_start|>'), tokenizer('<|im_end|>'), 

# Chat Template Creation

In [ ]:
chat_template = """{%- for message in messages -%}
    {%- if message['role']=='user' -%}
        {{- '<|im_start|>'+'User: '+message['content']+'<|im_end|>' -}}
    {%- elif message['role']=='assistant' -%}
        {{- '<|im_start|>'+'Assistant: '+message['content']+'<|im_end|>' -}}
    {%- elif message['role']=='system' -%}
            {{- '<|im_start|>'+'System: '+message['content']+'<|im_end|>' -}}
    {%- endif -%}
    {%- endfor -%}"""

tokenizer.chat_template = chat_template

# Data Pre-processing

In [ ]:
def combine_instruction_n_context(example):
    example['instruction'] = example['instruction']+"\n"*random.randint(0,3)+example['input']
    return example

In [ ]:
combined_input_dataset = dataset.map(combine_instruction_n_context)
combined_input_dataset

In [ ]:
dataset = combined_input_dataset.map(encode_example, remove_columns=['input'])
dataset

In [ ]:
dataset_splits = dataset['train'].train_test_split(test_size=0.05, seed=75)
dataset_splits

In [ ]:
print(dataset_splits['train'][0]['instruction'])
print("==="*10)
print(dataset_splits['train'][0]['output'])

In [ ]:
len(dataset['train'])

In [ ]:
example = {"messages":[{"role":"system",'content':'hi'},{"role":"user",'content':'hello'},{"role":"assistant",'content':'bye'}]}
print(tokenizer.apply_chat_template(example['messages'], tokenize=False), tokenizer.apply_chat_template(example['messages']))

In [ ]:
SYSTEM_PROMPT = """You are an helpful assistant. 
Your name is 'AARSON V1'. You follow user instructions clearly. 
You will use your knowledge and the provided context i.e., if there is any from the user, to help the user as much as you can."""

In [ ]:
def pre_process_data(example):

    example['example'] = {'messages':[{'role':'system', 'content':SYSTEM_PROMPT},
                           {'role':'user', 'content':example['instruction']},
                           {'role':'assistant', 'content':example['output']}]}
    return example

In [ ]:
dataset = dataset_splits.map(pre_process_data)
dataset, dataset['train'][0]['output'], dataset['train'][0]['example']

In [ ]:
sample = dataset['train'][0]['example']
inputs = tokenizer.apply_chat_template(sample['messages'], tokenize=True)['input_ids'][:-1]
labels = tokenizer.apply_chat_template(sample['messages'], tokenize=True)['input_ids'][1:]
print(tokenizer.decode(inputs))
print("-"*20)
print(tokenizer.decode(labels))

In [ ]:
print(tokenizer.apply_chat_template(dataset['train'][0]['example']['messages'], tokenize=False))
print("-"*30)
print(dataset['train'][0]['output']+"<|im_end|>")

# Dataset And DataLoader Creation

In [ ]:
from torch.utils.data import Dataset, DataLoader

MAX_LEN = 1024 # Kaggle GPU can't handel more

class MyDataset(Dataset):
    def __init__(self, data, tokenizer):
        super().__init__()
        self.data = data
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        model_input = self.data[idx]['example']
        model_label = self.data[idx]['output']+'<|im_end|>'

        tokenized_input = {}
        tokenized_output = {}

        # converting everything including user question and response into tokens
        input_tokens = self.tokenizer.apply_chat_template(model_input['messages'], tokenize=True, return_tensors="pt")
        tokenized_input['input_ids'] = input_tokens['input_ids'].squeeze()[:-1] # taking everything except last input token id
        tokenized_input['attention_mask'] = input_tokens['attention_mask'].squeeze()[:-1] # taking everything except last input token id attention mask
        
        output_tokens = self.tokenizer(model_label, padding='max_length', padding_side='left', max_length=input_tokens['input_ids'].shape[1], tokenize=True, return_tensors="pt")
        tokenized_output['input_ids'] = output_tokens['input_ids'].squeeze()[1:] # taking everything except first response token id
        tokenized_output['attention_mask'] = output_tokens['attention_mask'].squeeze()[1:] # taking everything except first response token id attention mask

        return tokenized_input, tokenized_output

train_dataset = MyDataset(dataset['train'], tokenizer)
eval_dataset = MyDataset(dataset['test'], tokenizer)

In [ ]:
print("Tokenizer padding side:", train_dataset.tokenizer.padding_side)

inputs, labels = train_dataset[0]

print("Input:")
print(tokenizer.decode(inputs["input_ids"]))

print("Labels:")
print(tokenizer.decode(
    labels["input_ids"][labels["attention_mask"].bool()]
))

In [ ]:
sample = train_dataset[0]
print(sample[0]['input_ids'].shape, sample[1]['input_ids'].shape)

print(tokenizer.decode(sample[0]['input_ids'].squeeze()))
print(tokenizer.decode(sample[1]['input_ids'].squeeze()))

# Custom Collate Function for Dynamic Padding and Truncation

In [ ]:
def custm_collate_function(batch):
    input_ids = [item[0]['input_ids'][:MAX_LEN] for item in batch]
    attention_mask = [item[0]['attention_mask'][:MAX_LEN] for item in batch]
    
    label_input_ids = [item[1]['input_ids'].squeeze()[:MAX_LEN] for item in batch]
    label_attention_mask = [item[1]['attention_mask'].squeeze()[:MAX_LEN] for item in batch]

    padded_inputs = tokenizer.pad({'input_ids':input_ids, 'attention_mask':attention_mask}, return_tensors='pt', padding=True)
    batch_sample = padded_inputs['input_ids'][0]
    # print(batch_sample.shape)

    padded_labels = tokenizer.pad({'input_ids':label_input_ids, 'attention_mask':label_attention_mask}, return_tensors='pt', padding='max_length', max_length=batch_sample.shape[0])
    # print(padded_inputs['input_ids'].shape)
    # print(padded_labels['input_ids'].shape)
    return padded_inputs.to(device), padded_labels.to(device)


In [ ]:
ex = dataset['train'][0]
inp = tokenizer.apply_chat_template(ex['example']['messages'], tokenize=False, return_tensors="pt")
print(inp)
print(ex['output']+'<|im_end|>')

In [ ]:
inp[-len(ex['output']+'<|im_end|>'):] == ex['output']+'<|im_end|>'

# Batching Train and Eval Datasets

In [ ]:
BATCH_SIZE = 4 

train_data_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=custm_collate_function)
eval_data_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=custm_collate_function)

In [ ]:
for idx, d in enumerate(eval_data_loader):
    print(f"input:\n{d[0]['input_ids'][0]}")
    print()
    print(f"label:\n{d[1]['input_ids'][0]}")
    print("="*30)
    if idx == 5:
        break

In [ ]:
for idx, d in enumerate(eval_data_loader):
    print(f"input:\n{d[0]['attention_mask'][0]}")
    print()
    print(f"label:\n{d[1]['attention_mask'][0]}")
    print("="*30)
    if idx == 1:
        break

In [ ]:
single_batch = next(iter(train_data_loader))
padded_inputs, padded_labels = single_batch
padded_inputs['input_ids'].shape, padded_labels['input_ids'].shape

# Initializing SmolLM2-135M in Float32 Precision

In [ ]:
from transformers import AutoModelForCausalLM #, AutoConfig

# model_config = AutoConfig.from_pretrained(model_repo_path)

# print(model_config.to_dict().keys())
# model_config.attention_dropout = 0.1
# print(model_config.rms_norm_eps)
#print(model_config.attention_dropout)

model = AutoModelForCausalLM.from_pretrained(model_repo_path, cache_dir="model/", dtype=torch.float32).to(device)

# Sample Model Fwd Pass

In [ ]:
model.eval()

with torch.inference_mode():
    # with torch.autocast(device, dtype=torch.float16):
        preds = model(padded_inputs["input_ids"])

preds

In [ ]:
print("Model dtype:", next(model.parameters()).dtype)
print("Logits dtype:", preds.logits.dtype)

# Calculate Loss Function

In [ ]:
def calculate_loss(loss_fn, preds_logits, labels, labels_attention_mask):
    labels = labels.clone()
    labels = labels.masked_fill(labels_attention_mask == 0, -100)

    loss = loss_fn(
        preds_logits.reshape(-1, preds_logits.shape[-1]),
        labels.reshape(-1),
    )

    return loss.reshape(labels.shape)

# Hyper parameters

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss(reduction='none', ignore_index=-100)
optim = torch.optim.AdamW(params=model.parameters(), lr=3e-5)
max_eval_iters = len(eval_data_loader)
eval_interval = 1500

# Step Function

In [ ]:
def step(model, inputs, labels, loss_fn, optim):
    optim.zero_grad(set_to_none=True)
    model.train()
    preds = model(inputs['input_ids'], inputs['attention_mask'])
    loss = calculate_loss(
    loss_fn,
    preds.logits,
    labels["input_ids"],
    labels["attention_mask"],)
    valid_tokens = labels["attention_mask"].bool()
    loss = loss[valid_tokens].mean()
    loss.backward()
    optim.step()
    return loss.item()

# Model Eval Function

In [ ]:
average_losses = {'train':[], 'eval':[]}
def eval_model(epoch, model, data_loaders, loss_fn, max_eval_iters):
    losses = {'train':0, 'eval':0}
    model.eval()
    for dataset, data_loader in data_loaders.items():
        for iter_, (inputs, labels) in enumerate(data_loader, start=1):
            with torch.no_grad():
                preds = model(inputs['input_ids'], inputs['attention_mask'])
                loss = calculate_loss(loss_fn, preds.logits, labels["input_ids"], labels["attention_mask"],)
                valid_tokens = labels["attention_mask"].bool()
                batch_loss = loss[valid_tokens].mean().item()
                losses[dataset] += batch_loss
            if iter_ == max_eval_iters:
                break
        losses[dataset] = losses[dataset]/max_eval_iters
        average_losses[dataset].append(losses[dataset])

    print(f"Epoch {epoch} => Train loss: {losses['train']} | Eval loss: {losses['eval']}")

# Model Train Function

In [ ]:
import time

data_loaders = {'train':train_data_loader, 'eval':eval_data_loader}

def train(epochs, model, data_loaders, loss_fn, optim, max_eval_iters):
    for epoch in range(epochs):
        average_epoch_loss = 0
        epoch_start = time.time()
        iter_times = []
        for iter_, (inputs, labels) in enumerate(data_loaders['train'], start=1):
            iter_start = time.time()
            if iter_%eval_interval == 0:
                eval_model(epoch, model, data_loaders, loss_fn, max_eval_iters)
            # if iter_==2:
            #     break
            loss = step(model, inputs, labels, loss_fn, optim)
            average_epoch_loss += loss
            iter_end = time.time()
            iter_times.append(iter_end-iter_start)

        epoch_end = time.time()
        print(f"Average epoch loss: {average_epoch_loss/len(data_loaders['train'])} | Epoch Time: {(epoch_end-epoch_start)/(60*60)} hrs.| Avg Iter Time: {sum(iter_times)/len(iter_times)} secs")

# Model Training

In [ ]:
epochs=2
train(epochs, model, data_loaders, loss_fn, optim, max_eval_iters)

# Loss Curves

In [ ]:
import matplotlib.pyplot as plt

plt.plot(average_losses['train'], marker='x', color='red', label='train')
plt.plot(average_losses['eval'], marker='.', color='blue', label='eval')
plt.legend()
plt.show()

# Saving Model Checkpoint

In [ ]:
out_dir='./saved_checkpoint'

model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)
torch.save(optim.state_dict(), f"{out_dir}/optim.pt")
print("Checkpoint Saved...")

# Build Chat Messages Helper Function

In [ ]:
def build_chat_messages(instruction, tokenize=True):
    tail = "<|im_start|>Assistant:"
    messages = {"messages":[{'role':'system', 'content':SYSTEM_PROMPT},
               {'role':'user','content':instruction}]}
    if tokenize:
        tokenized = tokenizer.apply_chat_template(messages['messages'], tokenize=tokenize, truncation=True, max_length=MAX_LEN, return_tensors="pt")
        tokenized_tail = tokenizer(tail, return_tensors="pt")
        
        return {'input_ids':torch.cat((tokenized['input_ids'], tokenized_tail['input_ids']), dim=-1),
                'attention_mask':torch.cat((tokenized['attention_mask'], tokenized_tail['attention_mask']), dim=-1)}
    else:
        return tokenizer.apply_chat_template(messages['messages'], tokenize=tokenize, truncation=True, max_length=MAX_LEN, return_tensors="pt")+tail

In [ ]:
tokenizer.decode(build_chat_messages(dataset['test']['instruction'][0], True)['input_ids'])[0] == build_chat_messages(dataset['test']['instruction'][0], False)

# Generate Function

In [ ]:
def generate(model, instruction, return_only_answer=False, max_new_tokens=None):
    input_ids = build_chat_messages(instruction, True)['input_ids'].to(device)
    if max_new_tokens is not None:
        output_ids = model.generate(input_ids, max_new_tokens=max_new_tokens, stop_strings=["<|im_end|>"],   # a list is the documented form
    tokenizer=tokenizer,).squeeze()
    else:  
        output_ids = model.generate(input_ids, stop_strings="<|im_end|>").squeeze()
    if return_only_answer:
        output_ids = output_ids[input_ids.shape[-1]:]
    return tokenizer.decode(output_ids)

# Loading Base Model For Comparision

In [ ]:
scratch_model_repo_name = r"HuggingFaceTB/SmolLM2-135M"

scratch_model = AutoModelForCausalLM.from_pretrained(scratch_model_repo_name, cache_dir="scratch/", dtype=torch.float32).to(device)

In [ ]:
import random

MAX_NEW_TOKENS = 300
for _ in range(5):
    random_index = random.randint(0, len(dataset['test']))
    instruction = dataset['test']['instruction'][random_index]

    print(instruction)
    print("-"*30)
    print("Finetuned Model:")
    print(generate(model, instruction, return_only_answer=True, max_new_tokens=MAX_NEW_TOKENS))
    print()
    print("="*30)
    print("Base Model:")
    print(generate(scratch_model, instruction, return_only_answer=True, max_new_tokens=MAX_NEW_TOKENS))
    print("*"*30)

# Push Model To Hub

In [ ]:
# model.push_to_hub("AJ700/instruct_finetuned_smollm_v1")
# tokenizer.push_to_hub("AJ700/instruct_finetuned_smollm_v1")

# Some Failing Instrucions For Better Improvement Later

In [ ]:
failing_instructions=["Where should I fly fish for trout in Washington state?",
                     "Classify each of the following as Toy or Tool: Garden Shovel, Golf Club, Baseball Bat, Hammer, Saw.",
                      """Given these paragraphs about Hydrogen Storage, what have been the two main historical forms of storage?


Several methods exist for storing hydrogen. These include mechanical approaches such as using high pressures and low temperatures, or employing chemical compounds that release H2 upon demand. While large amounts of hydrogen are produced by various industries, it is mostly consumed at the site of production, notably for the synthesis of ammonia. For many years hydrogen has been stored as compressed gas or cryogenic liquid, and transported as such in cylinders, tubes, and cryogenic tanks for use in industry or as propellant in space programs. Interest in using hydrogen for on-board storage of energy in zero-emissions vehicles is motivating the development of new methods of storage, more adapted to this new application. The overarching challenge is the very low boiling point of H2: it boils around 20.268 K (−252.882 °C or −423.188 °F). Achieving such low temperatures requires expending significant energy.

https://en.wikipedia.org/wiki/Hydrogen_storage""",
                      """Make me a bullet list of things to bring camping""",
                      "Classify each of the Pokémon characters into different types.",
                      
                     ]

In [ ]:
MAX_NEW_TOKENS = 300
for i in range(len(failing_instructions)):
    instruction = failing_instructions[i]
    # instruction = "Where should I fly fish for trout in Washington state?"
    # instruction = "what is the whether today in vijayawada?"
    print(instruction)
    print("-"*30)
    print("Finetuned Model:")
    print(generate(model, instruction, return_only_answer=True, max_new_tokens=MAX_NEW_TOKENS))
    print()
    print("="*30)
    print("Base Model:")
    print(generate(scratch_model, instruction, return_only_answer=True, max_new_tokens=MAX_NEW_TOKENS))
    print("*"*30)